# 09 — V3-B FULL Validation (Standalone)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/09_full_validation_standalone.ipynb)

This notebook is fully standalone and does **not** depend on the Part A tuning cells or on `COMMON`.

Frozen final method: **V3-B, 3 guidance steps, LR=0.02**. It runs the full common validation set, quality metrics, frozen-YOLO detection metrics, and visual saved/lost examples.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import os, sys, subprocess, json, hashlib
import numpy as np, pandas as pd, torch
from PIL import Image

PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345

DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
HR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'HR'
ORIG_SR=DATASET_ROOT/'difiisr_baseline'/'valid'/'SR_difiisr_train_degradation_v2'
ORIG_RESULTS=DATASET_ROOT/'difiisr_baseline'/'valid'/'results_difiisr_train_degradation_v2'
YOLO_WEIGHTS=DATASET_ROOT/'detector'/'weights'/'thermal_drone_yolo11n_corrected_v1_best.pt'
FULL_ROOT=DATASET_ROOT/'target_aware_difiisr'/'v3b_full_validation_v1'
FULL_ROOT.mkdir(parents=True, exist_ok=True)

project_dir=Path('/content/target-aware-infrared-sr')
difiisr_dir=Path('/content/DifIISR')

def run(cmd,cwd=None,env=None):
    print('$',' '.join(map(str,cmd)),flush=True)
    subprocess.run(list(map(str,cmd)),cwd=cwd,env=env,check=True)

if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
else:
    run(['git','fetch','origin'],cwd=project_dir)
    run(['git','checkout','main'],cwd=project_dir)
    run(['git','pull','--ff-only','origin','main'],cwd=project_dir)

if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir)
run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)

ANN_DIR=project_dir/'annotations'/'corrected_yolo'/'valid'/'labels'
ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv'])
    run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'pyiqa==0.1.12','basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown'])
run([sys.executable,'-m','pip','install','-q','ultralytics'])

weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True)
if not (weights/'DifIISR.pth').exists(): run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',weights/'DifIISR.pth'])
if not (weights/'autoencoder_vq_f4.pth').exists(): run([PY,'-m','gdown','https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth','-O',weights/'autoencoder_vq_f4.pth'])

assert LR_DIR.exists() and HR_DIR.exists() and ORIG_SR.exists() and ANN_DIR.exists() and YOLO_WEIGHTS.exists()
print('✓ setup ready | GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


In [ ]:
# Build the full common validation manifest.
FULL_MANIFEST=FULL_ROOT/'full_common_manifest.csv'
EXCLUDED=FULL_ROOT/'excluded_images.csv'
rows=[]; excluded=[]
for ip in sorted(p for p in LR_DIR.iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg'}):
    with Image.open(ip) as im: w,h=im.size
    lp=ANN_DIR/f'{ip.stem}.txt'
    is_bg=(not lp.exists()) or lp.stat().st_size==0
    rec=dict(image=ip.name,pilot_group='full_background' if is_bg else 'full_target',label_path=str(lp),is_background=is_bg,lr_width=w,lr_height=h,up_width=w*4,up_height=h*4)
    if (w,h)==(480,270) or (w,h)==(270,480): excluded.append(rec)
    else: rows.append(rec)
full_df=pd.DataFrame(rows)
pd.DataFrame(excluded).to_csv(EXCLUDED,index=False)
full_df.to_csv(FULL_MANIFEST,index=False)
print('Full common images:',len(full_df))
print('Target images:',int((~full_df.is_background).sum()),'| Background:',int(full_df.is_background.sum()),'| Excluded:',len(excluded))


In [ ]:
# FINAL CONFIG — standalone, no COMMON dependency.
FINAL_CONFIG=dict(
    seed=SEED,
    pilot_n_target_images=int((~full_df.is_background).sum()),
    pilot_n_background_images=int(full_df.is_background.sum()),
    small_target_definition='upsampled pre-encoder bbox width<=24 AND height<=24',
    core=24,outer=48,top_k=15,sigma_floor=0.013458,
    blob_k=1.5,soft_t=0.25,soft_min_support=3.0,
    scr_gate=1.8,scr_gate_temp=0.25,
    nd_gate=0.40,nd_gate_temp=0.07,nd_floor_weight=0.50,
    max_candidates=48,proposal_nms=7,
    scr_goal=2.5,goal_temp=0.25,
    guidance_steps=3,guidance_lr=0.02,
    lambda_fid=10.0,lambda_bg=20.0,
    difiisr_commit=DIFIISR_COMMIT,
    primary_selection_metric='GT-matched recall @ IoU>=0.25 on <=24x24 targets',
    selected_from_pilot='s3_lr002')
FINAL_RUN_ID='selected_s3_lr002'
FINAL_RUN_ROOT=FULL_ROOT/FINAL_RUN_ID
FINAL_RUN_ROOT.mkdir(parents=True,exist_ok=True)
FINAL_CONFIG_PATH=FINAL_RUN_ROOT/'config.json'
FINAL_CONFIG_PATH.write_text(json.dumps(FINAL_CONFIG,indent=2))
print(json.dumps(FINAL_CONFIG,indent=2))
print('Output:',FINAL_RUN_ROOT)


In [ ]:
# FULL inference — resume-safe.
RUNNER=project_dir/'scripts'/'v3_recall_guidance_runner.py'
DIFIISR_CONFIG=difiisr_dir/'configs'/'DifIISR_test.yaml'
cmd=[PY,str(RUNNER),'--manifest',str(FULL_MANIFEST),'--lr',str(LR_DIR),'--out',str(FINAL_RUN_ROOT),'--config',str(FINAL_CONFIG_PATH),'--difiisr-config',str(DIFIISR_CONFIG)]
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait(): raise RuntimeError('Full inference stopped. Completed images are saved; rerun this cell to resume.')
print('✓ full inference complete')


In [ ]:
# Quality metrics — Original DifIISR vs BASE vs V3-B selected.
QUALITY_SCRIPT=project_dir/'scripts'/'v3_full_quality_eval.py'
QUALITY_OUT=FINAL_RUN_ROOT/'quality_eval'
ORIG_CSV=ORIG_RESULTS/'valid_difiisr_baseline.csv'
env=os.environ.copy(); env['MPLBACKEND']='Agg'
cmd=[PY,str(QUALITY_SCRIPT),'--hr',str(HR_DIR),'--original',str(ORIG_SR),'--base',str(FINAL_RUN_ROOT/'BASE'),'--guided',str(FINAL_RUN_ROOT/'V3B_pooledSCR_ND'),'--out',str(QUALITY_OUT)]
if ORIG_CSV.exists(): cmd += ['--original-csv',str(ORIG_CSV)]
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait(): raise RuntimeError('Quality evaluation stopped. CSV checkpoints are saved; rerun this cell to resume.')
print('✓ quality metrics complete')


In [ ]:
# Frozen-YOLO full evaluation + saved/lost visual examples.
YOLO_SCRIPT=project_dir/'scripts'/'v3_full_yolo_eval.py'
YOLO_OUT=FINAL_RUN_ROOT/'yolo_full_eval'
cmd=[sys.executable,str(YOLO_SCRIPT),'--manifest',str(FULL_MANIFEST),'--original',str(ORIG_SR),'--base',str(FINAL_RUN_ROOT/'BASE'),'--guided',str(FINAL_RUN_ROOT/'V3B_pooledSCR_ND'),'--weights',str(YOLO_WEIGHTS),'--out',str(YOLO_OUT),'--iou-match','0.25','--fa-conf','0.25']
proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout: print(line,end='',flush=True)
if proc.wait(): raise RuntimeError('YOLO evaluation stopped. Detection caches are saved; rerun this cell to resume.')
print('✓ YOLO evaluation complete')
print('Visual examples:',YOLO_OUT/'visual_examples')


In [ ]:
# Final comparison table.
quality=pd.read_csv(QUALITY_OUT/'quality_all_methods.csv')
qsum=quality.groupby('method')[['psnr','ssim','lpips','clipiqa','musiq','niqe']].mean().reset_index()
det=pd.read_csv(YOLO_OUT/'detection_summary.csv')
final=det.merge(qsum,on='method',how='left')
display(final)
final.to_csv(FULL_ROOT/'FINAL_COMPARISON.csv',index=False)
print('Saved final table:',FULL_ROOT/'FINAL_COMPARISON.csv')
print('Saved/lost:',YOLO_OUT/'paired_saved_lost.csv')
print('Visual examples:',YOLO_OUT/'visual_examples')
